<a href="https://colab.research.google.com/github/MuhammadShayan8401/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadShayan8401/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research Question

Can historical search-performance and content signals identify content pages at risk of experiencing a **≥20% decline in organic-search impressions over the following 45 days**, so that limited content-review capacity can be prioritized?

### Decision Supported

The goal is not to automatically change or optimize content. The model is intended to provide a **directional risk signal** that helps an SEO or content team decide **which pages should receive human review first**.

Each content item is evaluated using information available from its historical performance. The resulting decline-risk score can be used to prioritize pages for investigation, such as reviewing recent search performance, content relevance, freshness, engagement, or content depth.

The final decision remains with a human reviewer. The system does not automatically publish, delete, redirect, rewrite, or modify content based solely on the model output.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Dataset Release

This study uses the public-safe, pseudonymized FlyRank ML Internship dataset, release flyrank_pseudonymized_warehouse_release_v20260703, exported on July 3, 2026. The release contains approximately 79 million rows of search-performance and content-related data representing roughly 70 clients.

Tables Used

The analysis uses the historical search-performance and query-level data needed to construct content-level features and future decline labels. The main signals include organic-search impressions, clicks, search position, query visibility, content characteristics, engagement signals, and content-age/freshness measures.

The modeling dataset was constructed at the content-item level from these underlying warehouse tables. Features were calculated from historical observations before the prediction period, while the target was calculated from the subsequent observation window.

Date Windows

Each modeling example uses two consecutive 45-day periods:

Previous 45 days: historical observation window used to construct model features.
Following 45 days: future window used only to determine whether the content item experienced a meaningful decline.

A content item is labeled as declining when its summed impressions in the future 45-day period are at least 20% lower than its summed impressions in the preceding 45-day period:

future_impressions < 0.80 × previous_impressions

The underlying daily time-series data used in this release runs through June 30, 2026. The release was exported on July 3, 2026, with an intentional freshness lag.

Exclusions

The analysis excludes information that would not be appropriate for a public research artifact or that could introduce leakage into the prediction task. In particular, raw client names, domains, URLs, private search queries, credentials, and other identifying or sensitive information are not used or exposed in the paper.

Future-period performance is also excluded from feature construction. It is used only for defining the outcome label, ensuring that the model does not receive information about the period it is intended to predict.

Rows without sufficient historical evidence to construct the required observation-window features or future outcome were excluded from the modeling dataset.

Public-Safety

The released data is pseudonymized and intended for public-safe analysis. This paper reports aggregate methodology, model behavior, and anonymized examples rather than exposing client identities, private queries, domains, URLs, credentials, or other potentially identifying information.

The resulting analysis should therefore be interpreted as a research and decision-support exercise over pseudonymized search-performance data, not as a disclosure of individual client performance.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*


### Assumptions

The study assumes that historical search-performance and content signals contain enough information to provide a useful directional indication of future visibility decline. The prediction task is treated as a classification problem rather than a causal analysis: model coefficients and feature relationships are interpreted as predictive associations, not evidence that changing a feature will cause performance to improve.

The intended use is prioritization of limited human review capacity. A model score is therefore treated as a decision-support signal rather than an autonomous content or SEO decision.

### Features

Features were constructed using information available from the historical observation window. The feature set combines search-performance, content, engagement, and freshness-related signals.

Examples include:

* Historical impressions and search performance
* Average search position
* Number of days with impressions
* Number of days with sessions
* Content age
* Days since the most recent update
* Word and character counts
* Scroll/engagement signals
* Query visibility and query-breadth measures
* Other historical content-performance indicators available before the prediction window

The final ML-08 modeling dataset contained 30,000 observations and 52 columns, with 21 numeric features used by the Logistic Regression model.

### Label Definition

The target variable is `is_declining`.

For each content item, impressions are aggregated over two consecutive 45-day periods. An item is labeled as declining when impressions in the future period are at least 20% lower than impressions in the preceding period:

`is_declining = 1 if future_impressions < 0.80 × previous_impressions`

Otherwise, the observation is labeled `0`.

This definition converts the business question into a measurable binary prediction task: identify pages that subsequently experience a substantial decline in organic-search impressions.

### Baseline

The model is evaluated against the existing Week-4/ML-07 heuristic baseline rather than against an arbitrary random benchmark.

The baseline assigns review priority using a weighted combination of historical visibility, query breadth, and CTR opportunity. Its decision logic uses approximately 50% historical visibility, 30% query breadth, and 20% CTR opportunity, with the resulting recommendation represented by the `VISIBILITY_REVIEW` reason code and `REVIEW_AND_REFRESH` action.

The baseline is important because the purpose of the experiment is not simply to produce a predictive model, but to determine whether the learned model provides useful prioritization compared with an existing rule-based approach.

### Validation Design

The Logistic Regression model was evaluated using a client-grouped holdout split with `GroupShuffleSplit`, grouping observations by `client_hash_id`.

This prevents observations from the same client from being distributed arbitrarily across the training and test sets, providing a more realistic test of whether the learned relationships generalize beyond the clients represented in training.

The resulting dataset contained 23,837 training observations and 6,163 held-out test observations across 32 clients. The observed declining rate in the modeling dataset was approximately 54.2%.

The primary comparison metric is **Precision@50**: among the 50 highest-priority predictions, the proportion that actually belong to the declining class.

This metric reflects the operational decision being supported: if review capacity is limited, how useful is the ranked list of highest-risk content items?

### Leakage Checks

Feature construction was restricted to information available before the future prediction window. Future-period impressions were used only to construct the target label and were not included as model features.

Client grouping was applied during validation so that information from the same client was not intentionally split between training and held-out evaluation groups.

The target definition was also kept separate from feature construction: the future 45-day impression total determines `is_declining`, while the model receives historical signals from the preceding observation period.

These controls reduce the risk that the model is evaluated using information that would not have been available at prediction time.

### Model Interpretation

The final model is a Logistic Regression classifier. Its coefficients are used to understand which signals are associated with higher or lower predicted decline risk within the fitted model.

These associations should not be interpreted causally. For example, freshness-related variables may indicate that an item deserves investigation, but the model does not establish that updating a page will prevent a future decline.


## 4. Results (vs baseline)

### Evaluation Setup

The Logistic Regression model and the Week-4/ML-07 heuristic baseline were evaluated on the same held-out evaluation setting so that the comparison reflects the same prediction task and review-prioritization objective.

The primary metric is Precision@50, which measures how many of the 50 highest-priority content items are actually members of the declining class.

### Honest Comparison

| Approach                  | Precision@50 |   Difference vs. Baseline |
| ------------------------- | -----------: | ------------------------: |
| ML-07 heuristic baseline  |      **92%** |                         — |
| ML-08 Logistic Regression |      **78%** | **−14 percentage points** |

The Logistic Regression model achieved **78% Precision@50**, compared with **92% for the existing ML-07 baseline**. On this evaluation, the learned model therefore did **not** outperform the baseline for the top-50 prioritization task.

This result is important: the experiment does not provide evidence that replacing the existing heuristic with Logistic Regression improves top-ranked review precision.

### What the Model Still Provides

Although the model underperformed the baseline on Precision@50, it provides a learned and inspectable risk score that can be used to study relationships between historical content/search signals and subsequent visibility decline.

The model's strongest predictive associations included signals such as days with impressions, word count, days with sessions, character count, content age, freshness-related measures, historical impressions, average position, and engagement signals.

These relationships are useful for generating hypotheses about which characteristics may warrant further investigation, but they should not be interpreted as causal effects.

### Interpretation

The result suggests that the existing heuristic was more effective than the tested Logistic Regression model at identifying declining items within the top 50 predictions under this evaluation setup.

Accordingly, the appropriate conclusion is **not** that machine learning improved the review process. Instead, the experiment demonstrates that a relatively simple learned model can be evaluated against an existing operational heuristic and that, in this experiment, the heuristic remains the stronger top-50 prioritization method.

The model should therefore be treated as an experimental decision-support signal rather than a replacement for the baseline.

### Error Profile

On the held-out evaluation data, the model produced:

* **3,525 true positives**
* **1,508 false positives**
* **1,130 false negatives**

This error profile reinforces the need for human review. A predicted decline does not guarantee that a page will decline, and some declining pages are not captured by the model's highest-risk predictions.

### Result Summary

**Primary finding:** Logistic Regression achieved 78% Precision@50 versus 92% for the ML-07 baseline, underperforming the baseline by 14 percentage points.

**Decision implication:** The tested model should not replace the existing baseline based on this experiment alone.

**Research implication:** Historical search and content signals contain predictive information worth investigating, but additional feature engineering, validation, model comparison, and longitudinal testing would be required before claiming that a learned model provides superior prioritization.


## 5. Limitations

This study provides an evaluation of decline-risk prediction for a specific dataset, feature set, model, and validation design. The findings should therefore be interpreted as decision-support evidence rather than as proof of a generally superior production system.

### The model does not establish causality

The relationships learned by Logistic Regression are predictive associations, not causal effects. A feature associated with higher or lower decline risk does not mean that changing that feature will cause search performance to improve or decline.

In particular, freshness, content age, engagement, and content-depth signals should be treated as review hypotheses rather than prescriptions for action.

### The model does not outperform the baseline

Under the reported evaluation, Logistic Regression achieved **78% Precision@50**, while the ML-07 heuristic baseline achieved **92%**. Therefore, this study cannot claim that the machine-learning approach is better than the existing baseline for top-50 prioritization.

A future model would need to demonstrate improvement under a comparable and robust evaluation before replacement of the baseline could be justified.

### Limited validation scope

The experiment uses a client-grouped holdout evaluation with `GroupShuffleSplit`. Although grouping by client reduces the risk of client-level information overlap between training and testing, a single holdout evaluation does not establish performance across all future clients, time periods, or search environments.

The results should therefore not be interpreted as evidence of guaranteed performance on unseen future data.

### Temporal changes may affect performance

Search behavior, content inventories, ranking systems, user behavior, and other environmental factors can change over time. A relationship learned from the available historical period may weaken or change when the underlying search environment changes.

Continued monitoring and evaluation on later data would be required to determine whether the observed relationships remain useful.

### Feature and label limitations

The model depends on the available historical search, content, and engagement signals. Missing data, measurement differences, feature availability, or changes in how these signals are collected can affect model behavior.

The ≥20% impression-decline threshold is also an analytical choice. Different thresholds or prediction horizons could produce different labels, rankings, and conclusions.

### No autonomous content actions

The system is not designed to automatically publish, delete, redirect, rewrite, or modify content. A high-risk score indicates that an item may deserve investigation; it does not establish that an action should be taken.

Final decisions require human review and additional contextual evidence that is not represented by the model.

### Public-data constraints

The analysis uses a pseudonymized, public-safe dataset. Client identities, domains, URLs, private search queries, and other sensitive information are intentionally excluded from the paper. As a result, the analysis cannot provide client-specific recommendations or expose the underlying private context behind individual observations.

### What this work can reasonably claim

The study can claim that historical search-performance and content signals were used to construct and evaluate a directional decline-risk classifier, and that the tested Logistic Regression model achieved **78% Precision@50** under the reported evaluation while the existing baseline achieved **92%**.

It can also support further investigation into which historical signals are associated with subsequent visibility decline.

It **cannot** claim that the model causes improved SEO outcomes, prevents future declines, generalizes universally to unseen search environments, or is superior to the existing baseline.


## 6. Ranked Recommendations

The model output is converted into a ranked action playbook designed to help content teams prioritize limited human-review capacity. Recommendations combine the model's decline-risk signal with supporting evidence from historical content and search-performance features.

The playbook intentionally stops short of autonomous content changes. A recommendation identifies **what should be investigated**, not what should automatically be changed.

### Ranked Action Playbook

| Rank | Signal / Condition                         | Recommended Action                                                                                             | Priority | Reason Code             |
| ---- | ------------------------------------------ | -------------------------------------------------------------------------------------------------------------- | -------- | ----------------------- |
| 1    | High decline-risk with supporting evidence | Investigate the page for potential visibility/performance decline and determine whether a refresh is warranted | High     | `HIGH_DECLINE_RISK`     |
| 2    | Strong freshness/update signal             | Review whether the content requires updating or additional freshness work                                      | High     | `FRESHNESS_SIGNAL`      |
| 3    | Search-performance or engagement signal    | Review engagement and historical search-performance indicators                                                 | Medium   | `ENGAGEMENT_SIGNAL`     |
| 4    | Content-depth signal                       | Review content depth, relevance, and whether the page adequately serves its intended search need               | Medium   | `CONTENT_DEPTH_SIGNAL`  |
| 5    | Model probability close to 0.50            | Escalate for human review because the model is uncertain                                                       | Low      | `UNCERTAIN_MODEL_SCORE` |
| 6    | Insufficient supporting evidence           | Continue monitoring rather than taking an immediate action                                                     | Low      | `INSUFFICIENT_EVIDENCE` |

### Recommended Review Sequence

**1. Investigate high-risk items first.**
Items with high predicted decline risk and supporting historical evidence receive the highest review priority. The reviewer should inspect the underlying search-performance and content signals before deciding whether any intervention is appropriate.

**2. Check freshness and update signals.**
Where freshness-related signals contribute to the recommendation, reviewers should determine whether the content is outdated, incomplete, or otherwise in need of review. A freshness signal is a reason to investigate, not evidence that updating the page will necessarily improve performance.

**3. Examine engagement and search performance.**
Reviewers should examine historical impressions, position, sessions, engagement, and related signals to understand whether the decline-risk score is supported by broader performance evidence.

**4. Review content depth and relevance.**
Where content-depth signals are prominent, reviewers should assess whether the page remains sufficiently comprehensive and relevant to its intended search need.

**5. Escalate uncertain predictions.**
Predictions near the model's decision boundary should not be treated as confident recommendations. These cases are better handled through human review or monitoring.

**6. Monitor when evidence is insufficient.**
When the available evidence does not justify an intervention, the appropriate recommendation is to monitor rather than force an action.

### Human-in-the-Loop Guardrails

The playbook is explicitly designed for **decision support**. The following actions are outside the scope of the system and should not be performed automatically from the model output:

* Automatically publishing content
* Automatically deleting pages
* Automatically creating redirects
* Automatically rewriting content
* Automatically changing titles or metadata
* Making other high-impact SEO decisions without human review

The model therefore answers **“which items should we investigate first?”** rather than **“what should we change automatically?”**

### Operational Interpretation

The ranked recommendations provide a practical bridge between model predictions and content-review workflows. Because the Logistic Regression model did not outperform the existing ML-07 baseline on Precision@50, the playbook should be used alongside existing prioritization logic rather than presented as a proven replacement.

The highest-value output of the system is consequently a structured queue of **review priorities, evidence signals, and suggested investigation paths**, with the final decision remaining under human control.


## 7. Artifacts the paper embeds

The paper embeds a compact set of reproducible tables, figures, and decision-support outputs. Each artifact is derived from the analysis described in this notebook and is intended to make the methodology and results inspectable.

### 1. Model vs. Baseline Comparison

**Artifact:** `work/figures/ml08_precision_at_50_comparison.png`

A direct comparison of Precision@50 for the ML-08 Logistic Regression model and the ML-07 heuristic baseline.

| Approach                  | Precision@50 |
| ------------------------- | -----------: |
| ML-07 heuristic baseline  |          92% |
| ML-08 Logistic Regression |          78% |

**Interpretation:** The Logistic Regression model is 14 percentage points below the existing baseline on the reported top-50 prioritization metric.

### 2. Model Error Profile

The paper reports the held-out prediction error counts:

| Outcome             | Count |
| ------------------- | ----: |
| Correct predictions | 3,525 |
| False positives     | 1,508 |
| False negatives     | 1,130 |

This table provides additional context for interpreting the model beyond its single headline Precision@50 result.

### 3. Ranked Action Queue

**Artifact:** `work/outputs/ranked_content_action_queue.csv`

The action queue converts model predictions and supporting evidence into ranked review recommendations. It includes reason codes such as:

* `HIGH_DECLINE_RISK`
* `FRESHNESS_SIGNAL`
* `ENGAGEMENT_SIGNAL`
* `CONTENT_DEPTH_SIGNAL`
* `UNCERTAIN_MODEL_SCORE`
* `INSUFFICIENT_EVIDENCE`

The queue is intended for human review prioritization and does not represent an autonomous content-action system.

### 4. Action-Playbook Evaluation Receipt

**Artifact:** `work/metrics/ml10_playbook_receipts.json`

The playbook receipt provides a machine-readable record of the action-playbook evaluation and supporting checks. It is included to make the transformation from model output to ranked recommendations auditable.

### 5. Model Signal Summary

The paper summarizes the strongest Logistic Regression coefficients to make the model interpretable. The most prominent signals include:

| Feature                  | Coefficient |
| ------------------------ | ----------: |
| `days_with_impressions`  |      0.7208 |
| `word_count`             |      0.5799 |
| `days_with_sessions`     |     −0.4996 |
| `char_count`             |     −0.3645 |
| `age_tier_order`         |     −0.1671 |
| `days_since_last_update` |      0.1571 |
| `content_age_days`       |     −0.1551 |
| `impressions_prev_30d`   |      0.1514 |
| `avg_position`           |     −0.1396 |
| `scroll_rate`            |      0.1279 |

These coefficients describe associations within the fitted model and should not be interpreted as causal effects.

### Artifact Reproducibility

All analysis artifacts are generated from the project notebook and stored under the repository's `work/` directory. The paper links the relevant notebook, figures, metrics, and output artifacts so that readers can inspect how the reported results and recommendations were produced.

### Public-Safety Note

Artifacts embedded or linked from the paper must remain public-safe. They must not expose client names, domains, URLs, private search queries, credentials, or other identifying information. Where individual examples are shown, only anonymized identifiers or row-level references should be used.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
